# 03 — Scaled v3 experiment analysis

Europarl scaled v3: 12,000 train / 1,500 validation / 1,500 test sentence pairs. This notebook reads saved predictions and metrics without generating new translations. Use validation to inspect choices and the held-out test for final reporting. Results from the v2 pilot belong in the separate notebook 02.


In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
DATASET = "europarl_scaled_v3"
SPLIT = "validation"  # Change to "test" to view the already-frozen final evaluation.
assert SPLIT in {"validation", "test"}
DATA_DIR = PROJECT_ROOT / "data" / "processed" / DATASET
assert DATA_DIR.is_dir(), f"Missing dataset: {DATA_DIR}"
ARMS = {
    "A": "sentence_lora_balanced_scaled_v3",
    "B": "context_lora_balanced_scaled_v3",
    "C": "sentence_lora_balanced_scaled_v3_llm",
    "D": "context_lora_balanced_scaled_v3_llm",
}
KEYS = ["pair", "doc_id", "src_sent_id", "tgt_sent_id"]
print(f"Project: {PROJECT_ROOT} | Dataset: {DATASET} | Split: {SPLIT}")


## Dataset provenance

Verify the exact CSV and builder manifest for this split. Predictions in A/B/C/D must align on the same sentence keys and source/reference text.


In [ ]:
manifest = json.loads((DATA_DIR / "summary.json").read_text(encoding="utf-8"))
data_path = DATA_DIR / f"{SPLIT}.csv"
import hashlib
actual_hash = hashlib.sha256(data_path.read_bytes()).hexdigest()
assert manifest["csv_sha256"][SPLIT] == actual_hash, "Dataset differs from summary.json"
truth = pd.read_csv(data_path)
assert truth["split"].eq(SPLIT).all()
assert not truth.duplicated(KEYS).any(), "Duplicate example keys in dataset"
display(truth.groupby("pair").agg(examples=("src", "size"), documents=("doc_id", "nunique")))
print(f"SHA256 {SPLIT}: {actual_hash}")


## Four-arm comparison

A/B prediction tables come from mBART evaluation; C/D come from LLM post-editing. Scores use the saved SacreBLEU and chrF metrics with one consistent reference per example. The code verifies both dataset hash provenance and row alignment before displaying a model.


In [ ]:
reports, predictions, missing = {}, {}, []
for arm, run in ARMS.items():
    pred_path = PROJECT_ROOT / "outputs" / "predictions" / f"{run}_{SPLIT}.csv"
    metric_path = PROJECT_ROOT / "outputs" / "metrics" / f"{run}_{SPLIT}_metrics.json"
    if not pred_path.is_file() or not metric_path.is_file():
        missing.append(f"{arm}: {pred_path.name} / {metric_path.name}")
        continue
    report = json.loads(metric_path.read_text(encoding="utf-8"))
    if arm in "AB":
        assert report.get("dataset_sha256") == actual_hash, f"{arm}: metric dataset hash mismatch"
        assert report.get("split") == SPLIT, f"{arm}: wrong split"
    else:
        assert report.get("split") == SPLIT and not report.get("smoke_test"), f"{arm}: wrong split or smoke result"
    table = pd.read_csv(pred_path)
    assert len(table) == len(truth), f"{arm}: example count mismatch"
    assert not table.duplicated(KEYS).any(), f"{arm}: duplicate keys"
    aligned = truth[KEYS + ["src", "tgt", "previous_src"]].merge(
        table[KEYS + ["src", "tgt", "previous_src", "prediction"]],
        on=KEYS, how="left", validate="one_to_one", suffixes=("_gold", "_out"),
    )
    assert aligned["prediction"].notna().all(), f"{arm}: missing predictions"
    for field in ("src", "tgt", "previous_src"):
        assert aligned[f"{field}_gold"].equals(aligned[f"{field}_out"]), f"{arm}: {field} mismatch"
    reports[arm], predictions[arm] = report, aligned["prediction"].reset_index(drop=True)
print("Available arms:", ", ".join(reports) or "none yet")
if missing:
    print("Pending outputs:")
    for entry in missing:
        print(" -", entry)


In [ ]:
rows = []
for arm, report in reports.items():
    for pair in ("en-de", "en-fr"):
        pair_report = report["pairs"][pair]
        score = pair_report if arm in "AB" else pair_report["postedited"]
        rows.append({"arm": arm, "pair": pair, "N": pair_report["num_examples"],
                     "BLEU": score["sacrebleu"], "chrF": score["chrf"]})
    pair_rows = rows[-2:]
    rows.append({"arm": arm, "pair": "macro", "N": sum(r["N"] for r in pair_rows),
                 "BLEU": sum(r["BLEU"] for r in pair_rows) / 2,
                 "chrF": sum(r["chrF"] for r in pair_rows) / 2})
score_table = pd.DataFrame(rows, columns=["arm", "pair", "N", "BLEU", "chrF"])
if score_table.empty:
    print("Predictions and metrics for the chosen split are pending.")
else:
    display(score_table.round(3))
    macro = score_table[score_table["pair"] == "macro"].set_index("arm")
    display(macro[["BLEU", "chrF"]].plot.bar(rot=0, figsize=(7, 4), title=f"{SPLIT}: macro scores").figure)
    plt.close("all")


## Paired changes and qualitative examples

Compare translations for the **same** rows. Sentence-level chrF identifies candidates for manual inspection; it does not prove that a model resolved a pronoun or improved discourse. Review the source, preceding source and both translations before writing such claims in the report.


In [ ]:
from sacrebleu.metrics import CHRF
chrf_metric = CHRF()
reference = truth["tgt"].tolist()
comparisons = {"context B − A": ("A", "B"), "LLM C − A": ("A", "C"),
               "LLM D − B": ("B", "D"), "context D − C": ("C", "D")}
summary = []
example_tables = {}
for name, (before, after) in comparisons.items():
    if before not in predictions or after not in predictions:
        continue
    old, new = predictions[before].tolist(), predictions[after].tolist()
    delta = [chrf_metric.sentence_score(b, [ref]).score -
             chrf_metric.sentence_score(a, [ref]).score
             for a, b, ref in zip(old, new, reference)]
    review = truth[KEYS + ["previous_src", "src", "tgt"]].copy()
    review[f"prediction_{before}"] = old
    review[f"prediction_{after}"] = new
    review["delta_sentence_chrF"] = delta
    example_tables[name] = review
    summary.append({"comparison": name, "changed": sum(a != b for a, b in zip(old, new)),
                    "helped": sum(d > 1e-9 for d in delta), "hurt": sum(d < -1e-9 for d in delta),
                    "tied": sum(abs(d) <= 1e-9 for d in delta),
                    "mean_sentence_chrF_delta": sum(delta) / len(delta)})
if summary:
    display(pd.DataFrame(summary).round(3))
    for name, table in example_tables.items():
        print(f"{name} — five largest positive changes")
        display(table.nlargest(5, "delta_sentence_chrF"))
        print(f"{name} — five largest negative changes")
        display(table.nsmallest(5, "delta_sentence_chrF"))
else:
    print("At least two complete arms are needed for paired examples.")


## COMET

Read saved reference-based COMET results for A/B/C/D on the selected split. Small descriptive mean differences do not by themselves establish statistical significance or a causal context effect.


In [ ]:
comet_path = PROJECT_ROOT / "outputs" / "metrics" / f"comet_{SPLIT}_scaled_v3_metrics.json"
if comet_path.is_file():
    comet_report = json.loads(comet_path.read_text(encoding="utf-8"))
    assert comet_report.get("dataset") == "scaled_v3"
    assert comet_report.get("split") == SPLIT
    assert comet_report["inputs"]["dataset"]["sha256"] == actual_hash
    comet_rows = []
    for arm, details in comet_report["arms"].items():
        for pair, metrics in details["pairs"].items():
            comet_rows.append({"arm": arm, "pair": pair, "COMET": metrics["comet_mean"]})
        comet_rows.append({"arm": arm, "pair": "macro",
                           "COMET": details["macro_average_comet"]})
    display(pd.DataFrame(comet_rows).round(4))
    display(pd.DataFrame([{"comparison": name, "COMET delta": value["macro_comet_delta"]}
                          for name, value in comet_report["comparisons"].items()]).round(4))
else:
    print(f"COMET pending: {comet_path.name}")


## Decoding ablation: Top-k and Top-p

Sampling was evaluated on **validation only**. The beam setting was frozen before the v3 test run. A positive COMET change with falling BLEU/chrF is reported as mixed evidence. The Qwen reranker does not see the gold reference.


In [ ]:
if SPLIT == 'validation':
    decoding_root = PROJECT_ROOT / 'outputs' / 'decoding' / 'scaled_v3'
    folders = {
        'Top-k': decoding_root / 'validation_top_k_k50_n3_t0p8_seed42',
        'Top-p': decoding_root / 'validation_top_p_p0p9_n3_t0p8_seed42',
    }
    decoding_rows = []
    for name, folder in folders.items():
        path = folder / 'metrics.json'
        if not path.is_file():
            print(f'Pending {name}: {path}')
            continue
        result = json.loads(path.read_text(encoding='utf-8'))
        assert result['config']['dataset_sha256'] == actual_hash
        assert result['config']['split'] == 'validation' and not result['smoke_test']
        for pair, values in result['pairs'].items():
            decoding_rows.append({'method': name, 'pair': pair, 'N': values['num_examples'],
                                  'changed': values['changed'],
                                  'beam BLEU': values['beam_bleu'],
                                  'selected BLEU': values['reranked_bleu'],
                                  'beam chrF': values['beam_chrf'],
                                  'selected chrF': values['reranked_chrf']})
    if decoding_rows:
        display(pd.DataFrame(decoding_rows).round(3))
    comet_sampling_path = decoding_root / 'validation_comet_comparison' / 'metrics.json'
    if comet_sampling_path.is_file():
        sampling_comet = json.loads(comet_sampling_path.read_text(encoding='utf-8'))
        assert sampling_comet['provenance']['dataset'] == actual_hash
        comet_rows = []
        for method, value in sampling_comet['summary'].items():
            for pair, item in value['pairs'].items():
                comet_rows.append({'method': method, 'pair': pair, 'COMET': item['comet_mean'],
                                   'delta vs beam': item['mean_delta_vs_beam']})
            comet_rows.append({'method': method, 'pair': 'macro', 'COMET': value['macro_comet'],
                               'delta vs beam': value['macro_delta_vs_beam']})
        display(pd.DataFrame(comet_rows).round(4))
    else:
        print('Sampling COMET pending: run scripts/evaluate_decoding_comet_v2.py')
else:
    print('Sampling ablation was selected on validation; the test split is not loaded here.')


## Interpretation

Read the per-pair rows before interpreting equal-pair macro averages. Differences between validation and test or between Europarl v2 and v3 are not controlled model comparisons across datasets. Blinded AI judgments are exploratory error analysis, not human evaluation. A preceding sentence may clarify a referent, but a score change alone does not prove that context caused the translation improvement.


## Balanced versus unweighted LoRA

These four models share the v3 dataset and evaluation split. The unweighted models are an ablation, kept separate from the A/B/C/D post-edit experiment. Check dataset hashes before comparing scores.


In [ ]:
ABLATED = {
    "sentence_balanced": "sentence_lora_balanced_scaled_v3",
    "sentence_unweighted": "sentence_lora_unweighted_scaled_v3",
    "context_balanced": "context_lora_balanced_scaled_v3",
    "context_unweighted": "context_lora_unweighted_scaled_v3",
}
ablation_rows = []
for label, run in ABLATED.items():
    path = PROJECT_ROOT / "outputs" / "metrics" / f"{run}_{SPLIT}_metrics.json"
    if not path.is_file():
        print("Pending ablation metrics:", path.name)
        continue
    result = json.loads(path.read_text(encoding="utf-8"))
    assert result["split"] == SPLIT and result["dataset_sha256"] == actual_hash
    for pair in ("en-de", "en-fr"):
        score = result["pairs"][pair]
        ablation_rows.append({"model": label, "pair": pair,
                              "BLEU": score["sacrebleu"], "chrF": score["chrf"]})
    ablation_rows.append({"model": label, "pair": "macro",
                          "BLEU": result["macro_average_sacrebleu"],
                          "chrF": result["macro_average_chrf"]})
if ablation_rows:
    display(pd.DataFrame(ablation_rows).round(3))


## Blinded error review (AI-assisted)

Rows compare changed predictions, including separately selected COMET extremes. Only the random cue/non-cue rows are exploratory samples of changed outputs. These labels were produced by AI and must not be called human judgments.


In [ ]:
if SPLIT == "validation":
    review_path = PROJECT_ROOT / "outputs/analysis/abcd_validation_scaled_v3/manual_review/ai_review_summary.json"
    if review_path.is_file():
        manual = json.loads(review_path.read_text(encoding="utf-8"))
        assert manual["dataset"] == "scaled_v3" and manual["split"] == "validation"
        comparison_rows = []
        for comparison, subsets in manual["comparisons"].items():
            for pair in ("en-de", "en-fr", "both"):
                values = subsets["random"][pair]
                comparison_rows.append({"comparison": comparison, "pair": pair,
                                        "N changed sampled": values["n"],
                                        "after better": values["meaning"]["after"],
                                        "before better": values["meaning"]["before"],
                                        "tie": values["meaning"]["tie"]})
        display(pd.DataFrame(comparison_rows))
    else:
        print("AI review summary pending:", review_path)
else:
    print("Manual review is validation-only; no test labels were used to select models.")
